# Machine Learning Exercise

Now it's time to practice what you have seen in the previous notebooks. Your task for today is to download the data from the database and train a model in order to predict the binary label in this heart-related teaching dataset. 

![](https://www.nicepng.com/png/detail/397-3975460_disease-high-quality-png-heart-disease-cartoon-png.png)

This exercise is not a clinically validated risk model. Confirm the target definition before giving it a medical interpretation.

## Task

1. Inspect the five tables in the `heart` schema and join them using the patient key. Verify that the join has one row per patient.
2. Confirm target coding and which source tags are eligible; exclude identifiers and provenance tags from model inputs.
3. Reserve a stratified test set before exploration. Use the remaining data for EDA and model selection.
4. Build a pipeline that imputes, encodes, scales, and fits logistic regression. Fit preprocessing separately within each training fold.
5. Compare a simple baseline, scalers, and hyperparameters using validation data or cross-validation, never the test set.
6. Freeze the selected pipeline and evaluate it once on the test set.

**Deliver:** a reproducible notebook with the join, data-quality decisions, validation comparison, final classification report/confusion matrix, and a short explanation of limitations.

## Keep in Mind

- Choose the positive class and success metric before tuning. Accuracy alone may hide important mistakes.
- Use identical folds and the same metric to compare candidates.
- Keep missing feature values for an imputer instead of silently removing difficult test cases.
- Numerical category codes are not necessarily continuous measurements.
- Tune with a modest randomized or grid search before increasing the search budget.
- Do not infer clinical meaning or data provenance from correlations alone.

## Data Overview

The database is a teaching adaptation. Confirm its label definition and category coding using the database documentation: the column name `heart_attack` alone does not establish whether it represents disease, past events, or a recoded label.

| Column | Meaning / treatment |
|---|---|
| patient_id | Join key, not a model input. Check one row per patient. |
| age | Age in years. |
| sex | Coded sex variable; check the database's coding. |
| chest_pain_type | Categorical chest-pain code; do not assume equally spaced values. |
| resting_blood_pressure | Resting blood pressure in mm Hg. |
| fasting_blood_sugar | Indicator for fasting blood sugar above 120 mg/dl; verify coding. |
| thal | Categorical test-result code; verify normal/defect/missing codes. |
| serum_cholestoral | Serum cholesterol in mg/dl; spelling follows the schema. |
| resting_electrocardiographic_results | Categorical resting ECG result. |
| maximum_heartrate_achieved | Maximum heart rate during an exercise test. |
| exercise_induced_angina | Categorical exercise-induced angina indicator. |
| oldpeak | Exercise-induced ST depression relative to rest. |
| slope_of_the_peak_exercise_st_segment | Categorical ST-segment slope code. |
| number_of_major_vessels_colored_by_flourosopy | Vessel count; verify missing/sentinel codes. |
| real_data | Provenance tag, not a predictive feature. |
| heart_attack | Binary exercise label; confirm what 0 and 1 mean before interpretation. |

[UCI Heart Disease](https://archive.ics.uci.edu/dataset/45/heart+disease) describes **presence of heart disease**, not future heart-attack risk. Its original target is 0-4; public adaptations often binarize or recode it. Do not assume this database uses the original codes. This exercise is not a clinical decision tool.

The categorical descriptions are still useful: chest-pain categories describe typical/atypical angina, non-anginal pain, or no symptoms; ECG categories describe normal or abnormal resting results; ST-slope categories describe upsloping, flat, or downsloping segments. Their integer mappings must come from this database's documentation, not an assumed mapping from another dataset.

## Import

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

# Add modeling imports as you use them.

## Getting the Data

The data for this exercise is stored in our postgres database in the schema `heart`. The different features are split thematically into five different tables. Your first task will be to have a look at the tables (e.g. in DBeaver) and figure out a way to join the information you need. As soon as you're happy with your query, you can use the following code cells to import the data into this notebook. 

In previous notebooks you've seen two different approaches to import data from a database into a notebook. The following code uses `sqlalchemy` in combination with pandas `pd.read_sql()`. Copy `.env.example` to `.env` in this repository root and fill in `DATABASE`, `USER_DB`, `PASSWORD`, `HOST`, and `PORT` using the provided credentials. The notebook builds the connection URL automatically; enter the password as-is, without URL encoding. Existing `.env` files with a complete `DB_STRING` are also supported. Change `query_string` to your own query. Never commit credentials.

The supplied query only previews ten rows from `heart.patient`. Replace it with your complete join before modeling. The connection is closed after reading; the DataFrame remains available.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "solutions":
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / ".env")

DB_STRING = os.getenv("DB_STRING")
if not DB_STRING or "<" in DB_STRING:
    # Build the URL from the fields in .env; URL.create handles special characters.
    field_names = ("USER_DB", "PASSWORD", "HOST", "DATABASE")
    fields = {name: os.getenv(name) for name in field_names}
    if any(not value or "<" in value for value in fields.values()):
        raise ValueError(
            "Set DB_STRING or USER_DB, PASSWORD, HOST, DATABASE and PORT in .env."
        )
    try:
        port = int(os.getenv("PORT") or "5432")
    except ValueError:
        raise ValueError("PORT in .env must be an integer.") from None
    DB_STRING = URL.create(
        "postgresql+psycopg2",
        username=fields["USER_DB"],
        password=fields["PASSWORD"],
        host=fields["HOST"],
        port=port,
        database=fields["DATABASE"],
    )

db = create_engine(DB_STRING, connect_args={"connect_timeout": 10})

In [ ]:
# Preview only: replace with your five-table join before modeling.
query_string = "SELECT * FROM heart.patient LIMIT 10"
with db.connect() as connection:
    df_sqlalchemy = pd.read_sql(text(query_string), connection)
db.dispose()
df_sqlalchemy.head()

In [ ]:
# Optional: export the completed join to the Git-ignored data/ directory.
# Keep private data in the Git-ignored data/ directory, never in the repository root.
# Save DataFrame as CSV after creating the data/ directory if needed:
# df_sqlalchemy.to_csv("data/heart_data.csv", index=False)

## 1. Check Your Completed Join

Replace the ten-row preview query with your five-table join before continuing. Check the columns, row count, missing patient keys, and repeated keys. Report exact duplicates and missing labels; do not silently discard conflicting rows for the same patient. Confirm the label and provenance rules using the course data documentation.

**Expected result:** a joined DataFrame with one row per eligible patient, plus a short record of exclusions and their reasons. Keep missing feature measurements for the pipeline's imputer.

In [ ]:
# Inspect df_sqlalchemy and check the join before creating your modeling DataFrame.
# Explain exclusions; retain patient_id for checks, not as a model feature.

## 2. Define the Target, Metric, and Split

Write what class 1 means and choose the main selection metric before tuning. Exclude `patient_id` and `real_data` from model inputs. Reserve a stratified test set; use only the remaining development data for exploration and selection.

You can use training-only cross-validation as in notebook 1, or add a separate validation set as in the worked solution. In either case, preprocessing belongs inside each CV training fold.

<img src="assets/heart-evaluation-protocol.svg" width="650" style="max-width: 100%;" alt="The worked solution uses training-only cross-validation, a separate validation set, and one final test evaluation.">

**Expected result:** named subsets, their sizes and class proportions, and a written evaluation plan. Keep the test set untouched until Step 6.

In [ ]:
# Define features and target, reserve test data, and record your selection metric.
# If using a separate validation set, split it from the development data.

## 3. Explore Training Data

Inspect types, missing measurements, class balance, category codes, and numerical ranges in training data only. Choose a few plots that inform a preprocessing decision. Confirm special missing-value codes before interpreting them as measurements.

**Expected result:** a concise data-quality summary and your numerical/categorical feature groups, with reasons for their treatment. Do not infer clinical meaning or verified provenance from a plot.

In [ ]:
# Explore only training rows and document the preprocessing choices they justify.

## 4. Establish a Baseline and Build a Pipeline

Choose a simple baseline appropriate to your metric, then build a `ColumnTransformer` and `Pipeline` with imputation, categorical encoding, a numerical scaler, and logistic regression. For coded categories such as chest-pain type, do not assume the integers represent equally spaced measurements.

**Expected result:** a baseline and a complete pipeline evaluated using your development-data protocol. Explain what the baseline predicts; high recall alone can hide many false positives.

In [ ]:
# Add modeling imports as needed, then build the baseline and full pipeline.
# Evaluate with development data only; do not fit preprocessing before CV.

## 5. Compare and Select Candidates

Compare unscaled, standardized, and min-max-scaled pipelines with the same metric and CV folds. Run a modest search on training folds; if you have a separate validation set, use it for the final candidate comparison. Include the baseline and keep the test scores out of this table.

**Expected result:** one comparison table and a selected configuration with a short justification. Report when tuning fails to improve on a simpler candidate.

In [ ]:
# Compare scalers and a small hyperparameter search under your chosen protocol.
# Freeze the selected recipe before looking at the test results.

## 6. Evaluate Once on the Test Set

First answer the three questions at the end of this notebook. With the configuration fixed, fit the complete pipeline on all development data, then evaluate once on the test set. A CV search with `refit=True` already refits on all the data passed to its `fit`; a separate validation subset is not included automatically.

**Expected result:** the final selection metric, classification report, and a confusion matrix with labeled axes. Explain false positives and false negatives using the confirmed label definition. Do not use the result to tune again.

In [ ]:
# Refit the frozen configuration if needed, then perform the final test evaluation.
# Keep final test results separate from your development-data comparison.

## 7. Explain Your Result

Summarize whether the selected model improves on the baseline under your chosen metric, what the errors mean, and two limitations of the data or evaluation. Distinguish dataset-label prediction from future heart-attack risk.

**Expected result:** a short conclusion accompanying your reproducible notebook. After completing it, compare your choices with the [worked solution](solutions/2_exercise_machine_learning.ipynb); different valid choices need not produce identical scores.

## Check Your Understanding

Before your final test evaluation, explain:

1. Why should `patient_id` be excluded from model inputs, even if it improves validation scores?
2. Why must the imputer be fitted inside each cross-validation training fold?
3. Why should you not choose between models using their test scores?

Explain your reasoning before consulting the worked solution.